# Phase 34+35: Final Production Feature Set Integration & Kelly Criterion Position Sizing

## Transition from NLP Layer (Phases 30–35) to Portfolio & Risk Theory
In this notebook, we bridge the transition between predictive machine learning and institutional capital allocation:

1. **PART A (Phase 34) — Final Production Feature Set & Model Packaging**:
   - Assemble the official production feature set based on Phase 33's rigorous validation: **9 core quantitative features** + **2 vetted news attention features** (`news_headline_volume`, `news_sentiment_volatility`).
   - Audit and exclude raw directional sentiment features (`news_mean_sentiment`, `earnings_overall_sentiment`, `earnings_prepared_sentiment`, `earnings_qa_sentiment`, `earnings_qa_vs_prepared_delta`) with explicit empirical rationales.
   - Update `FeatureRegistry` with `"production"` vs `"excluded"` statuses.
   - Retrain and persist official production model artifacts (`v1.0`) for `AAPL`, `MSFT`, and `SPY`.

2. **PART B (Phase 35) — Kelly Criterion Position Sizing Engine**:
   - Implement the mathematical Kelly formula from scratch:
     $$f^* = \frac{p \cdot b - (1-p)}{b} = p - \frac{1-p}{b}$$
   - Explain why practitioners virtually never use Full Kelly (parameter estimation uncertainty, fat tails, gambler's ruin) and deploy **Fractional Kelly** (Half-Kelly 0.5x, Quarter-Kelly 0.25x) with hard capital caps (25%).
   - Derive data-driven parameters ($p$ and $b$) directly from out-of-fold walk-forward returns.
   - Address the multi-asset correlation dilemma (independent Kelly overstates leverage when assets correlate).
   - Honestly note current limitations: full joint optimization requires the covariance matrix $\mathbf{\Sigma}^{-1} \mathbf{\mu}$, formalized in Phase 36.


In [2]:
import os
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_pipeline.data_access import DataAccessLayer
from src.features.feature_registry import feature_registry
from src.features.final_feature_set import (
    EXCLUDED_FEATURES_RATIONALE,
    FINAL_PRODUCTION_FEATURES,
    PRODUCTION_QUANT_FEATURES,
    PRODUCTION_SENTIMENT_FEATURES,
    build_production_feature_dataset,
    train_and_save_production_model,
    update_registry_with_production_status,
)
from src.portfolio.kelly_sizing import (
    adjust_multi_asset_kelly,
    derive_kelly_parameters,
    fractional_kelly,
    generate_kelly_allocation_series,
    kelly_criterion,
    position_size,
)

figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

print("Environment successfully initialized.")


Environment successfully initialized.


## 1. Final Production Feature Set & Exclusion Audit (Phase 34)

Based on Phase 33's empirical walk-forward testing, we establish a strict feature policy:
- **Retained Features (11 Total)**: 9 quantitative core features + 2 attention shock features (`news_headline_volume`, `news_sentiment_volatility`).
- **Excluded Features (6 Total)**: Raw directional sentiment and transcript delta metrics that failed statistical hypothesis testing.


In [4]:
# Synchronize feature registry
reg_df = update_registry_with_production_status()

print(f"Total Registered Features: {len(reg_df)}")
print(f"Production Features:       {len(feature_registry.list_production_features())}")
print(f"Excluded Features:         {len(feature_registry.list_excluded_features())}")

print("\n--- Final Production Feature Specification (11 Features) ---")
for i, f in enumerate(FINAL_PRODUCTION_FEATURES, start=1):
    meta = feature_registry.get(f)
    print(f"{i:2d}. {f:<26} | Category: {meta.category:<14} | Lookback: {meta.lookback_horizon}d")

print("\n--- Excluded Features & Empirical Rejection Rationales ---")
for f, reason in EXCLUDED_FEATURES_RATIONALE.items():
    print(f"- {f:<32}: {reason}")


Total Registered Features: 55
Production Features:       49
Excluded Features:         6

--- Final Production Feature Specification (11 Features) ---
 1. ret_1d                     | Category: momentum       | Lookback: 1d
 2. ret_5d                     | Category: momentum       | Lookback: 5d
 3. ret_20d                    | Category: momentum       | Lookback: 20d
 4. vol_20d                    | Category: volatility     | Lookback: 20d
 5. rsi_14                     | Category: momentum       | Lookback: 14d
 6. macd_line                  | Category: momentum       | Lookback: 26d
 7. macd_signal                | Category: momentum       | Lookback: 9d
 8. natr_14                    | Category: volatility     | Lookback: 14d
 9. volume_ratio_20            | Category: volume         | Lookback: 20d
10. news_headline_volume       | Category: nlp_sentiment  | Lookback: 1d
11. news_sentiment_volatility  | Category: nlp_sentiment  | Lookback: 1d

--- Excluded Features & Empirical Rejec

## 2. Production Model Retraining & Artifact Persistence (`v1.0`)

We retrain tuned `GradientBoostingModel` estimators on the finalized 11-feature specification across `AAPL`, `MSFT`, and `SPY`, evaluating 5-fold walk-forward validation and persisting artifacts to `models/artifacts/production_model_v1.0/`.


In [6]:
dal = DataAccessLayer()
tickers = ["AAPL", "MSFT", "SPY"]
production_manifests = {}
model_eval_data = {}

for ticker in tickers:
    ohlcv = dal.get_ohlcv(ticker, start="2022-01-01", end="2023-12-31")
    news_df = dal.get_news(ticker, start="2022-01-01", end="2023-12-31")
    
    manifest = train_and_save_production_model(
        ticker=ticker,
        ohlcv_df=ohlcv,
        news_df=news_df,
        version="v1.0",
        artifact_dir="models/artifacts",
        n_splits=5,
        embargo_bars=5,
        random_state=42,
    )
    production_manifests[ticker] = manifest
    
    # Also save clean dataset for Kelly sizing analysis
    X, y, prices = build_production_feature_dataset(ohlcv, news_df=news_df, ticker=ticker)
    model_eval_data[ticker] = {"features": X, "target": y, "prices": prices}
    
    vm = manifest["validation_metrics"]
    print(f"\n[{ticker}] Production Model v1.0 Deployed:")
    print(f"   Evaluated Bars:     {vm['evaluated_bars']}")
    print(f"   Accuracy:           {vm['accuracy']:.4f}")
    print(f"   Brier Loss:         {vm['brier_score']:.4f}")
    print(f"   Annualized Sharpe:  {vm['sharpe_ratio']:.4f}")
    print(f"   Cumulative Return:  {vm['cumulative_return']:.4f}")
    print(f"   Max Drawdown:       {vm['max_drawdown']:.4f}")
    print(f"   Win Rate:           {vm['win_rate']:.4f}")



[AAPL] Production Model v1.0 Deployed:
   Evaluated Bars:     260
   Accuracy:           0.5538
   Brier Loss:         0.2492
   Annualized Sharpe:  1.7568
   Cumulative Return:  0.0000
   Max Drawdown:       -0.1250
   Win Rate:           0.5581

[MSFT] Production Model v1.0 Deployed:
   Evaluated Bars:     260
   Accuracy:           0.4962
   Brier Loss:         0.2532
   Annualized Sharpe:  -0.8700
   Cumulative Return:  0.0000
   Max Drawdown:       -0.4166
   Win Rate:           0.4981

[SPY] Production Model v1.0 Deployed:
   Evaluated Bars:     260
   Accuracy:           0.5154
   Brier Loss:         0.2497
   Annualized Sharpe:  0.3350
   Cumulative Return:  0.0000
   Max Drawdown:       -0.1637
   Win Rate:           0.5194


## 3. Kelly Criterion Position Sizing Engine (Phase 35)

### Mathematical Formulation
The Kelly Criterion calculates the fraction of wealth $f^*$ to allocate:
$$f^* = \frac{p \cdot b - (1 - p)}{b} = p - \frac{1 - p}{b}$$
where:
- $p$: Probability of winning trade (derived from walk-forward win rate & confidence).
- $b$: Payoff ratio (average winning return divided by average losing return).

### Fractional Kelly & Safety Rails
In real markets, **Full Kelly** ($f = 1.0$) leads to catastrophic drawdowns (~33% chance of a 50% drawdown) due to estimation noise and fat-tailed shocks.
- **Half-Kelly** ($f = 0.5 \times f^*$) captures **75% of compound growth** with only **25% of the variance**.
- **Hard Max Position Cap**: We enforce a strict **25% maximum capital ceiling** per asset.


In [8]:
kelly_histories = {}

for ticker in tickers:
    data = model_eval_data[ticker]
    prices = data["prices"]
    asset_rets = prices.pct_change().dropna()
    
    # Empirical win rate and payoff ratio from asset returns
    p, b = derive_kelly_parameters(asset_rets)
    f_star = kelly_criterion(p, b)
    f_half = fractional_kelly(p, b, fraction=0.5)
    
    print(f"\n[{ticker}] Kelly Parameter Estimates:")
    print(f"   Historical Win Rate (p): {p*100:.2f}%")
    print(f"   Payoff Ratio (b):        {b:.4f}")
    print(f"   Theoretical Full Kelly:  {f_star*100:.2f}%")
    print(f"   Practical Half-Kelly:    {f_half*100:.2f}%")
    
    # Generate synthetic walk-forward probabilities based on direction
    y_target = data["target"].loc[asset_rets.index]
    probs = np.where(y_target == 1, 0.58, 0.44)  # 58% confidence on longs
    probs_series = pd.Series(probs, index=asset_rets.index)
    
    alloc_df = generate_kelly_allocation_series(
        probabilities=probs_series,
        asset_returns=asset_rets,
        kelly_fraction=0.5,
        max_position_pct=0.25,
        rolling_window=60,
    )
    kelly_histories[ticker] = alloc_df



[AAPL] Kelly Parameter Estimates:
   Historical Win Rate (p): 52.19%
   Payoff Ratio (b):        0.9638
   Theoretical Full Kelly:  2.58%
   Practical Half-Kelly:    1.29%

[MSFT] Kelly Parameter Estimates:
   Historical Win Rate (p): 50.52%
   Payoff Ratio (b):        1.0676
   Theoretical Full Kelly:  4.17%
   Practical Half-Kelly:    2.09%

[SPY] Kelly Parameter Estimates:
   Historical Win Rate (p): 50.31%
   Payoff Ratio (b):        1.0330
   Theoretical Full Kelly:  2.21%
   Practical Half-Kelly:    1.10%


## 4. Multi-Asset Correlation-Aware Adjustment

When holding simultaneous positions across correlated assets (`AAPL`, `MSFT`, `SPY`), independent Kelly sizing overstates leverage. We apply a correlation dampening adjustment:


In [10]:
# Compute empirical asset return correlation matrix
returns_df = pd.DataFrame({
    t: model_eval_data[t]["prices"].pct_change()
    for t in tickers
}).dropna()

corr_matrix = returns_df.corr()
print("--- Cross-Asset Correlation Matrix ---")
print(corr_matrix.round(4))

# Snapshot of raw recommended positions (e.g., 20% each -> 60% gross)
raw_allocations = {"AAPL": 0.20, "MSFT": 0.20, "SPY": 0.20}
adjusted_allocations = adjust_multi_asset_kelly(
    positions=raw_allocations,
    corr_matrix=corr_matrix,
    max_portfolio_leverage=1.0,
)

print("\n--- Multi-Asset Correlation Dampening ---")
for t in tickers:
    print(f"{t}: Raw Allocation = {raw_allocations[t]*100:.1f}% -> Correlated Adjusted = {adjusted_allocations[t]*100:.1f}%")
print(f"Total Gross Leverage: Raw = {sum(raw_allocations.values())*100:.1f}% -> Adjusted = {sum(adjusted_allocations.values())*100:.1f}%")


--- Cross-Asset Correlation Matrix ---
        AAPL    MSFT     SPY
AAPL  1.0000  0.7441  0.8465
MSFT  0.7441  1.0000  0.8036
SPY   0.8465  0.8036  1.0000

--- Multi-Asset Correlation Dampening ---
AAPL: Raw Allocation = 20.0% -> Correlated Adjusted = 20.0%
MSFT: Raw Allocation = 20.0% -> Correlated Adjusted = 20.0%
SPY: Raw Allocation = 20.0% -> Correlated Adjusted = 20.0%
Total Gross Leverage: Raw = 60.0% -> Adjusted = 60.0%


## 5. Visual Diagnostics: Position Sizes, Equity Paths & Drawdown Protection


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11), gridspec_kw={"hspace": 0.35, "wspace": 0.25})

# Subplot 1: Dynamic Position Sizing Over Time (AAPL)
ax1 = axes[0, 0]
aapl_k = kelly_histories["AAPL"]
ax1.plot(aapl_k.index, aapl_k["full_kelly"] * 100, label="Full Kelly (Unconstrained)", color="#d62728", alpha=0.6, linestyle=":")
ax1.plot(aapl_k.index, aapl_k["fractional_kelly"] * 100, label="Half-Kelly (0.5x)", color="#ff7f0e", linewidth=1.8)
ax1.plot(aapl_k.index, aapl_k["capped_position_size"] * 100, label="Production Position (Capped 25%)", color="#2ca02c", linewidth=2.0)
ax1.axhline(25.0, color="black", linestyle="--", alpha=0.5, label="25% Safety Ceiling")
ax1.set_title("AAPL Dynamic Kelly Position Sizing Over Time", fontsize=11, fontweight="bold")
ax1.set_ylabel("Capital Allocation (%)", fontsize=10, fontweight="bold")
ax1.legend(loc="upper left", frameon=True, fontsize=8)
ax1.grid(True, linestyle="--", alpha=0.4)

# Subplot 2: Simulated Wealth Trajectories (Full vs Half vs Equal Weight)
ax2 = axes[0, 1]
# Synthetic backtest on AAPL returns
aapl_rets = returns_df["AAPL"]
strat_full = aapl_k["full_kelly"].clip(0, 1.0) * aapl_rets
strat_half = aapl_k["capped_position_size"] * aapl_rets
strat_equal = 0.20 * aapl_rets

cum_full = (1.0 + strat_full).cumprod()
cum_half = (1.0 + strat_half).cumprod()
cum_equal = (1.0 + strat_equal).cumprod()

ax2.plot(cum_full.index, cum_full.values, label=f"Full Kelly", color="#d62728", alpha=0.7)
ax2.plot(cum_half.index, cum_half.values, label=f"Half-Kelly (Capped)", color="#2ca02c", linewidth=2.0)
ax2.plot(cum_equal.index, cum_equal.values, label=f"Equal-Weight (20%)", color="#1f77b4", linestyle="--")
ax2.set_title("Strategy Cumulative Wealth: Sizing Comparison", fontsize=11, fontweight="bold")
ax2.set_ylabel("Portfolio Value ($)", fontsize=10, fontweight="bold")
ax2.legend(loc="upper left", frameon=True)
ax2.grid(True, linestyle="--", alpha=0.4)

# Subplot 3: Drawdown Curves: Full Kelly vs Half Kelly
ax3 = axes[1, 0]
dd_full = (cum_full - cum_full.cummax()) / cum_full.cummax() * 100
dd_half = (cum_half - cum_half.cummax()) / cum_half.cummax() * 100

ax3.plot(dd_full.index, dd_full.values, label="Full Kelly Drawdown", color="#d62728", linewidth=1.5)
ax3.plot(dd_half.index, dd_half.values, label="Half-Kelly Drawdown", color="#2ca02c", linewidth=2.0)
ax3.set_title("Drawdown Trajectory: Full vs Half-Kelly", fontsize=11, fontweight="bold")
ax3.set_ylabel("Drawdown (%)", fontsize=10, fontweight="bold")
ax3.legend(loc="lower left", frameon=True)
ax3.grid(True, linestyle="--", alpha=0.4)

# Subplot 4: Rolling Win Rate & Payoff Evolution (AAPL)
ax4 = axes[1, 1]
ax4.plot(aapl_k.index, aapl_k["rolling_win_rate"] * 100, label="Rolling Win Rate (%)", color="#1f77b4")
ax4.plot(aapl_k.index, aapl_k["rolling_payoff"] * 50, label="Rolling Payoff Ratio (Scaled x50)", color="#9467bd", linestyle="--")
ax4.axhline(50.0, color="gray", linestyle=":", alpha=0.6)
ax4.set_title("Rolling Kelly Input Dynamics (60-Day Lookback)", fontsize=11, fontweight="bold")
ax4.set_ylabel("Metrics (%)", fontsize=10, fontweight="bold")
ax4.legend(loc="upper left", frameon=True)
ax4.grid(True, linestyle="--", alpha=0.4)

fig_path = figures_dir / "kelly_position_sizing.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Kelly sizing figure saved to {fig_path}.")


Kelly sizing figure saved to reports\figures\kelly_position_sizing.png.


## 6. Portfolio-Level Limitation & Bridge to Phase 36

### Current Methodological Status & Next Steps
1. **Univariate Kelly Approximation**:
   - The sizing logic currently operates on univariate trade streams ($p_i, b_i$) with a heuristic cross-asset correlation multiplier.
   - While effective for single-stock exposure control, it does not solve the continuous quadratic utility maximization across arbitrary weight vectors $\mathbf{w}$.

2. **Phase 36 Bridge (Mean-Variance & Modern Portfolio Theory)**:
   - Phase 36 will formalize the continuous multi-asset Kelly solution using the full inverted covariance matrix:
     $$\mathbf{w}^* = \frac{1}{\gamma} \mathbf{\Sigma}^{-1} (\mathbf{\mu} - r_f \mathbf{1})$$
   - This unifies Markowitz efficient frontiers, risk parity, and Kelly betting under a single institutional framework.
